# FaceForensics++ Explorer — Grupo 8 (Kaggle)
**Curso:** 1INF52 Deep Learning | **Tema:** Robustez de detectores de deepfakes

Este notebook descarga un subconjunto del dataset FaceForensics++ usando el script oficial,
extrae frames de los videos y muestra una cuadricula comparativa de caras **reales vs falsas**
de distintos metodos de manipulacion.

> **Requisito:** Haber recibido el correo de aprobacion de acceso a FF++.

---

### Como usar este notebook en Kaggle

1. Ir a [kaggle.com/code](https://www.kaggle.com/code) → **New Notebook**
2. Subir este `.ipynb` (File → Import Notebook)
3. En la barra lateral derecha, en **Settings**:
   - **Accelerator** → GPU P100 o T4 (para cuando entrenen el modelo)
   - **Internet** → **On** (necesario para descargar los videos de FF++)
4. Correr las celdas en orden

### Tip: guardar los datos como Dataset privado
Despues de descargar los videos por primera vez, pueden guardarlos como un **Dataset privado**
en Kaggle para no tener que re-descargar cada vez:

1. Correr la celda de descarga (seccion 2)
2. En la barra lateral derecha → **Output** → **New Dataset** → darle un nombre (ej: `ff-plus-plus-c23`)
3. En futuros notebooks, agregar ese dataset desde **Add data** en la barra lateral
4. Los archivos estaran en `/kaggle/input/ff-plus-plus-c23/` (o el nombre que le pongan)

## 1. Configuracion de rutas
Kaggle usa `/kaggle/working/` para archivos de salida y `/kaggle/input/` para datasets.

- Si es la **primera vez** (descarga desde cero): dejamos `USAR_DATASET_EXISTENTE = False`
- Si ya subieron los videos como dataset privado: cambien a `True` y pongan el nombre

In [ ]:
import os

# ============================================================
# CONFIGURACION — modificar segun su caso
# ============================================================
USAR_DATASET_EXISTENTE = False  # True si ya subieron los videos como dataset
NOMBRE_DATASET = 'ff-plus-plus-c23'  # nombre del dataset en Kaggle (si aplica)

NUM_VIDEOS = 10       # videos a descargar por categoria
COMPRESSION = 'c23'   # c23 = calidad alta (estandar), c40 = mas comprimido
SERVER = 'EU2'        # servidor aprobado en el correo de FF++
# ============================================================

if USAR_DATASET_EXISTENTE:
    BASE_DIR = f'/kaggle/input/{NOMBRE_DATASET}'
    print(f'Usando dataset existente en: {BASE_DIR}')
    if not os.path.exists(BASE_DIR):
        print('ERROR: No se encontro el dataset. Verificar:')
        print('  1. Que el nombre coincida con el dataset subido')
        print('  2. Que este agregado en "Add data" en la barra lateral')
else:
    BASE_DIR = '/kaggle/working/FF_data'
    print(f'Modo descarga: los datos se guardaran en {BASE_DIR}')
    print('Tip: al terminar, guarden el output como Dataset privado para no re-descargar.')

## 2. Descargar el script y los videos
Esta seccion solo corre si `USAR_DATASET_EXISTENTE = False`.

In [ ]:
if not USAR_DATASET_EXISTENTE:
    # El script oficial llega con la aprobacion de acceso a FF++ y NO se publica ni se enlaza aqui
    # (terminos de uso). Subanlo a Kaggle como Dataset privado: Add Input -> Upload.
    from pathlib import Path
    SCRIPT = next((p for p in Path('/kaggle/input').rglob('*.py')
                   if 'faceforensics' in p.name.lower() and 'download' in p.name.lower()), None)
    assert SCRIPT, 'Suban el script de descarga de FF++ como Kaggle Dataset privado'
    print(f'Script: {SCRIPT}\n')
    
    datasets_to_download = ['original', 'Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures']
    
    for ds in datasets_to_download:
        print(f'\n{"="*60}')
        print(f'Descargando: {ds} ({COMPRESSION}, {NUM_VIDEOS} videos)')
        print(f'{"="*60}')
        !echo "" | python {SCRIPT} {BASE_DIR} -d {ds} -c {COMPRESSION} -t videos -n {NUM_VIDEOS} --server {SERVER}
    
    print('\nDescarga completada!')
    print('\nRECORDATORIO: Guarden el output como Dataset privado (barra lateral → Output → New Dataset)')
else:
    print('Saltando descarga — usando dataset existente.')

## 3. Explorar la estructura de archivos

In [ ]:
for root, dirs, files in os.walk(BASE_DIR):
    level = root.replace(BASE_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if files:
        subindent = ' ' * 2 * (level + 1)
        shown = files[:5]
        for f in shown:
            print(f'{subindent}{f}')
        if len(files) > 5:
            print(f'{subindent}... y {len(files) - 5} archivos mas')

## 4. Extraer frames de los videos
De cada video tomamos algunos frames espaciados uniformemente.

In [ ]:
import cv2
import numpy as np
from glob import glob

datasets = ['original', 'Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures']


def extraer_frames(video_path, n_frames=5):
    """Extrae n_frames espaciados uniformemente de un video."""
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return []
    
    indices = np.linspace(0, total - 1, n_frames, dtype=int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames


def obtener_videos(dataset_name):
    """Busca los .mp4 descargados para un dataset dado."""
    if dataset_name == 'original':
        pattern = os.path.join(BASE_DIR, 'original_sequences', 'youtube',
                               COMPRESSION, 'videos', '*.mp4')
    else:
        pattern = os.path.join(BASE_DIR, 'manipulated_sequences', dataset_name,
                               COMPRESSION, 'videos', '*.mp4')
    return sorted(glob(pattern))


# Extraer 3 frames de los primeros 3 videos de cada dataset
frames_por_dataset = {}
for ds in datasets:
    videos = obtener_videos(ds)
    print(f'{ds}: {len(videos)} videos encontrados')
    ds_frames = []
    for v in videos[:3]:
        fs = extraer_frames(v, n_frames=3)
        if fs:
            ds_frames.append((os.path.basename(v), fs))
    frames_por_dataset[ds] = ds_frames

print('\nFrames extraidos correctamente!')

## 5. Visualizar: Real vs Fake
Cuadricula comparativa — cada fila es un dataset, cada columna un video distinto.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(datasets), 3, figsize=(15, 3 * len(datasets)))
fig.suptitle('FaceForensics++ — Muestras por dataset', fontsize=16, y=1.01)

for i, ds in enumerate(datasets):
    ds_frames = frames_por_dataset.get(ds, [])
    for j in range(3):
        ax = axes[i][j]
        if j < len(ds_frames) and len(ds_frames[j][1]) > 0:
            vid_name, frs = ds_frames[j]
            ax.imshow(frs[1])  # frame del medio
            if j == 0:
                label = 'REAL' if ds == 'original' else 'FAKE'
                color = 'green' if ds == 'original' else 'red'
                ax.set_ylabel(f'{ds}\n({label})', fontsize=11,
                              color=color, fontweight='bold')
            ax.set_title(vid_name[:15], fontsize=8)
        else:
            ax.text(0.5, 0.5, 'Sin datos', ha='center', va='center',
                    transform=ax.transAxes)
        ax.axis('off')

plt.tight_layout()
plt.show()

## 6. Zoom: Diferencias sutiles entre real y fake
Buscamos un par donde el video original y su version Deepfake coincidan para comparar frame a frame.

In [ ]:
orig_videos = obtener_videos('original')
df_videos = obtener_videos('Deepfakes')

# Los Deepfakes se nombran XXX_YYY.mp4 (target_source)
orig_ids = {os.path.basename(v).replace('.mp4', ''): v for v in orig_videos}

par_encontrado = None
for df_v in df_videos:
    df_name = os.path.basename(df_v).replace('.mp4', '')
    target_id = df_name.split('_')[0]
    if target_id in orig_ids:
        par_encontrado = (orig_ids[target_id], df_v, target_id, df_name)
        break

if par_encontrado:
    orig_path, df_path, orig_id, df_id = par_encontrado
    print(f'Par encontrado:')
    print(f'  Original: {orig_id}.mp4')
    print(f'  Deepfake: {df_id}.mp4')
    
    orig_frames = extraer_frames(orig_path, n_frames=5)
    df_frames = extraer_frames(df_path, n_frames=5)
    
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    fig.suptitle(f'Comparacion: Original ({orig_id}) vs Deepfake ({df_id})',
                 fontsize=14)
    
    for j in range(5):
        if j < len(orig_frames):
            axes[0][j].imshow(orig_frames[j])
        axes[0][j].axis('off')
        if j == 0:
            axes[0][j].set_ylabel('REAL', fontsize=13, color='green',
                                  fontweight='bold')
        
        if j < len(df_frames):
            axes[1][j].imshow(df_frames[j])
        axes[1][j].axis('off')
        if j == 0:
            axes[1][j].set_ylabel('FAKE', fontsize=13, color='red',
                                  fontweight='bold')
    
    plt.tight_layout()
    plt.show()
else:
    print('No se encontro un par original-deepfake coincidente.')
    print('Puede pasar si los IDs no coinciden en los 10 videos descargados.')

## 7. Estadisticas basicas de los videos

In [ ]:
print(f'{"Dataset":<20} {"Videos":<10} {"Resolucion":<15} {"FPS":<8} {"Frames prom.":<12}')
print('=' * 65)

for ds in datasets:
    videos = obtener_videos(ds)
    if not videos:
        print(f'{ds:<20} {0:<10} {"N/A":<15} {"N/A":<8} {"N/A":<12}')
        continue
    
    cap = cv2.VideoCapture(videos[0])
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    total_frames = []
    for v in videos:
        c = cv2.VideoCapture(v)
        total_frames.append(int(c.get(cv2.CAP_PROP_FRAME_COUNT)))
        c.release()
    
    avg_frames = int(np.mean(total_frames))
    cap.release()
    
    print(f'{ds:<20} {len(videos):<10} {f"{w}x{h}":<15} {fps:<8.1f} {avg_frames:<12}')

**Resultado de la exploracion** (ejecutada en Kaggle el 07/10/2026 con `NUM_VIDEOS = 10` por categoria):

| Categoria | Videos | Resolucion | FPS | Frames promedio |
|---|---|---|---|---|
| original | 10 | 1280x720 | 30 | 482 |
| Deepfakes | 10 | 1280x720 | 30 | 482 |
| Face2Face | 10 | 1280x720 | 30 | 482 |
| FaceSwap | 10 | 1280x720 | 30 | 442 |
| NeuralTextures | 10 | 1280x720 | 30 | 442 |

Las cuadriculas de frames real vs. falso se eliminaron del repositorio: los terminos de uso de FF++ no permiten redistribuir videos ni frames. Para verlas, ejecutar este notebook en Kaggle con acceso aprobado.

## 8. (Opcional) Descargar mas datos
Descomentar segun lo que necesiten para las siguientes fases del proyecto.

In [ ]:
# === DESCOMENTAR Y MODIFICAR SEGUN NECESIDAD ===

# Mas videos (50 en vez de 10)
# !echo "" | python {SCRIPT} /kaggle/working/FF_data -d Deepfakes -c c23 -t videos -n 50 --server EU2

# Compresion c40 para pruebas de robustez (Fase C)
# !echo "" | python {SCRIPT} /kaggle/working/FF_data_c40 -d Deepfakes -c c40 -t videos -n 10 --server EU2
# !echo "" | python {SCRIPT} /kaggle/working/FF_data_c40 -d original -c c40 -t videos -n 10 --server EU2

# FaceShifter (otro metodo de manipulacion)
# !echo "" | python {SCRIPT} /kaggle/working/FF_data -d FaceShifter -c c23 -t videos -n 10 --server EU2

print('Celda de descarga opcional. Descomentar lo que necesiten.')

## 9. Guardar como Dataset privado
Si descargaron los videos en este notebook y quieren reutilizarlos sin re-descargar:

1. En la barra lateral derecha de Kaggle → **Save Version** (boton azul arriba)
2. Seleccionar **Save & Run All** → **Save**
3. Una vez termine, ir a la pestaña **Output** del notebook
4. Click en **New Dataset** → nombre: `ff-plus-plus-c23` → **Create**
5. En el proximo notebook: **Add data** → buscar su dataset → listo, los videos estaran en `/kaggle/input/ff-plus-plus-c23/FF_data/`

Con esto, los notebooks futuros (entrenamiento de Xception, etc.) arrancan al toque sin esperar descargas.

---
**Grupo 8** — Ronaldo Segura, Diego Alvarez, Yaser Fernandez  
1INF52 Deep Learning · PUCP · 2026-2